# Flow Chart Pengerjaan:
1. Cover Image
2. DCT & Quantization
3. Block Smoothness Estimation & Sorting
4. Zigzag Scan
5. NACP Construction
6. Adaptive Hexagonal Payload Assignment
7. Hexagonal Turtle Shell Embedding
8. Stego DCT Coefficients
9. Entropy Coding
10. Stego Image

In [183]:
!pip install jpeglib numpy matplotlib opencv-python-headless scipy scikit-image seaborn pandas tqdm import-ipynb


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [184]:
from PIL import Image
from performance import psnr, fsi, ssim
from zigzag import zigzag, inverse_zigzag
from math import ceil, floor, log2, log10, sqrt
import copy
import cv2
import jpeglib
import numpy as np
import import_ipynb
import matplotlib.pyplot as plt
import turtleShell
import FrequencyDomain as FD

In [185]:
def change_image_QF(image_path, target_qf):
    im = jpeglib.read_dct(image_path)
    old_qt = im.qt[0]

    # From QF 100 to target QF
    dequantized = im.Y.astype(np.float64) * old_qt
    new_coefficients = np.round(dequantized / FD.custom_q_mat(target_qf)).astype(np.int16)
    
    # Update image object
    im.Y[:] = new_coefficients
    im.qt[0] = FD.custom_q_mat(target_qf)

    dequantized = im.Y.astype(np.float64) * FD.custom_q_mat(target_qf)
    im.Y[:] = np.round(dequantized / FD.custom_q_mat(100)).astype(np.int16)
    im.qt[0] = FD.custom_q_mat(100)
    
    ori_path =  image_path.split(".jpeg")[0]
    output_path = f"{ori_path}_qf{target_qf}.jpeg"
    im.write_dct(output_path)

In [186]:
def get_quantized_coefficients(image_path):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, _, _  = im.Y.shape
    sorted_coeffs = []
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block = im.Y[i, j]
            zigzag_coeffs = zigzag(block)
            sorted_coeffs.append(zigzag_coeffs)
    return sorted_coeffs

In [187]:
def get_compress_coeff(image_path, target_qf):
    im = jpeglib.read_dct(image_path)
    old_qt = im.qt[0]
    dequantized = im.Y.astype(np.float64) * old_qt
    new_coefficients = np.round(dequantized / FD.custom_q_mat(target_qf)).astype(np.int16)
    im.Y[:] = new_coefficients
    im.qt[0] = FD.custom_q_mat(target_qf)
    num_v_blocks, num_h_blocks, _, _ = im.Y.shape
    sorted_coeffs = []
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block = im.Y[i, j]
            zigzag_coeffs = zigzag(block)
            sorted_coeffs.append(zigzag_coeffs)
    return sorted_coeffs

In [188]:
def convert_tiff_to_jpeg(tiff_path, jpeg_path, quality=100):
    with Image.open(tiff_path) as img:
        rgb_img = img.convert('L')
        rgb_img.save(jpeg_path, 'JPEG', quality=quality, subsampling=0, optimize=False)
    print(f"Converted {tiff_path} to {jpeg_path} with quality {quality}")

In [189]:
def sort_smoothness(smoothness_list):
    smoothness_list.sort(key=lambda x: (-x[1], x[2]))
    return smoothness_list

In [190]:
def block_smoothness_estimation(image):
    im = jpeglib.read_dct(image)
    h, w, _, _ = im.Y.shape
    smoothness_block = []
    total_ec = 0
    total_zero_count = 0
    for i in range(h):
        for j in range(w):
            block = im.Y[i, j]
            print( block)
            block_1d = block.flatten()
            block_1d = block_1d[1:]  # AC coefficients 
            zero_count = np.sum(block_1d == 0)
            non_zero_sum = np.sum(abs(block_1d[block_1d != 0]))
            non_zero_indices = np.nonzero(block_1d)[0]
            capable_bits = 4 if zero_count == 0 else 3
            total_ec += len(non_zero_indices) // 2 * capable_bits
            total_zero_count += zero_count
            smoothness_block.append(((i, j), zero_count, non_zero_sum))

    print(f"Total blocks: {h * w}")
    print(f"Total embedding capacity (estimated): {total_ec} bits")
    print(f"Total zero count: {total_zero_count}")
    # return smoothness_block

def block_smoothness(image):
    im = jpeglib.read_dct(image)
    h, w, _, _ = im.Y.shape
    q_table = im.qt[0]
    smoothness_block = []
    smoothness_score = []
    for i in range(h):
        for j in range(w):
            block = im.Y[i, j]
            ac_block = block.copy()
            ac_block[0, 0] = 0
            z_k = np.sum(ac_block == 0)
            E_k = np.sum((ac_block != 0) * (q_table ** 2))
            S_k = z_k + float(z_k / E_k)
            smoothness_block.append(((i, j), z_k, E_k, S_k))
            smoothness_score.append(((i, j), S_k))
    
    print(f"Total blocks: {h * w}")
    print(smoothness_block)
    print(smoothness_score)
    return smoothness_block, smoothness_score

In [191]:
def invariant_ac_smoothness(image_path, threshold_eob):
    coeffs = get_quantized_coefficients(image_path)
    smoothness_block = []
    for idx in range(len(coeffs)):
        sum_z_k = 0
        sum_ac_k = 0
        for k in range(threshold_eob + 1, 64):
            if coeffs[idx][k] == 0:
                sum_z_k += 1
            sum_ac_k += abs(coeffs[idx][k])
        smoothness_block.append(((idx), sum_z_k, sum_ac_k))
    s_smoothness_block = sort_smoothness(smoothness_block)
    return smoothness_block, s_smoothness_block

In [192]:
def get_nacp(sorted_coefficients):
    valid_nacp = []
    for zigzag_coeff in sorted_coefficients:
        ac_coeffs = zigzag_coeff[1:] # AC coefficients
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]
        non_zero_ac = [ac_coeffs[i] for i in non_zero_indices]

        for i in range(0, len(non_zero_ac) - 1, 2):
            x = int(non_zero_ac[i])
            y = int(non_zero_ac[i+1])
            if x != 0 and y != 0:
                valid_nacp.append((x, y))

    return valid_nacp

def get_nacp_2(sorted_coefficients):
    valid_nacp = []
    for zigzag_coeff in sorted_coefficients:
        ac_coeffs = zigzag_coeff[1:] # AC coefficients
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) > 1)[0]
        non_zero_ac = [ac_coeffs[i] for i in non_zero_indices]

        for i in range(0, len(non_zero_ac) - 1, 2):
            x = int(non_zero_ac[i])
            y = int(non_zero_ac[i+1])
            if x != 0 and y != 0:
                valid_nacp.append((x, y))

    return valid_nacp

In [193]:
def replace_nacp(sorted_coefficients, nacp_coords):
    pair_index = 0

    for zigzag_coeff in sorted_coefficients:
        ac_coeffs = zigzag_coeff[1:]  
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]
        non_zero_ac = [ac_coeffs[i] for i in non_zero_indices]

        for idx in range(0, len(non_zero_ac) - 1, 2):
            if pair_index < len(nacp_coords):
                new_x, new_y = nacp_coords[pair_index]
                i1, i2 = non_zero_indices[idx], non_zero_indices[idx + 1]
                ac_coeffs[i1] = float(new_x)
                ac_coeffs[i2] = float(new_y)
                pair_index += 1
            else: break
        zigzag_coeff[1:] = ac_coeffs

    return sorted_coefficients

def replace_nacp_2(sorted_coefficients, nacp_coords):
    pair_index = 0
    for zigzag_coeff in sorted_coefficients:
        ac_part = zigzag_coeff[1:]
        rel_non_zero_indices = np.nonzero(np.abs(ac_part) > 1)[0]        
        for idx in range(0, len(rel_non_zero_indices) - 1, 2):
            if pair_index < len(nacp_coords):
                new_x, new_y = nacp_coords[pair_index]                
                zigzag_coeff[rel_non_zero_indices[idx] + 1] = float(new_x)
                zigzag_coeff[rel_non_zero_indices[idx+1] + 1] = float(new_y)
                pair_index += 1
            else: break
    return sorted_coefficients

In [194]:
def construct_stego_file(image_path, new_coeffs):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, v_block_size, h_block_size  = im.Y.shape
    idx = 0
    
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block_coeffs = new_coeffs[idx]
            block = inverse_zigzag(block_coeffs, v_block_size, h_block_size)
            im.Y[i, j] = block
            idx += 1

    output_path = "stego-images/stego_" + image_path.split("/")[-1]
    print(f"Image with secret data is saved to {output_path}")
    im.write_dct(output_path)

def construct_stego_file_2(image_path, new_coeffs, qf):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, v_block_size, h_block_size  = im.Y.shape
    idx = 0
    
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block_coeffs = new_coeffs[idx]
            block = inverse_zigzag(block_coeffs, v_block_size, h_block_size)
            im.Y[i, j] = block
            idx += 1

    if qf is not None:    
        dequantized = im.Y.astype(np.float64) * FD.custom_q_mat(qf)
        im.Y[:] = np.round(dequantized / FD.custom_q_mat(100)).astype(np.int16)
        im.qt[0] = FD.custom_q_mat(100)

    output_path = "stego-images/stego_" + image_path.split("/")[-1]
    print(f"Image with secret data is saved to {output_path}")
    im.write_dct(output_path)

In [195]:
def data_hiding_process(secret_data, nacp_coord="", mode="8N"):
    if secret_data == "": return nacp_coord
    bit = 3 if mode == "8N" else 4
    secret_data = secret_data + '\0'
    data_bin = ''.join(format(ord(c), '08b') for c in secret_data)
    lendata = len(data_bin)
    print(f"Secret Data: {secret_data}")
    print(f"Panjang Bit Secret Data: {lendata}")

    decimals = []
    for i in range(0, len(data_bin), bit):
        group = data_bin[i:i+bit].ljust(bit, '0')
        decimals.append(int(group, 2))

    _, shells, cell_to_shells = turtleShell.init(mode) 
    if len(decimals) > len(nacp_coord):
        print("Warning: Not enough NACP coordinates to embed all data.")
        
    for i in range(len(decimals)):
        x, y = nacp_coord[i]
        if turtleShell.get_hex_matrix_value(x, y, mode) == decimals[i]:
            nacp_coord[i] = (x, y)
        else:
            # shell_coords = turtleShell.get_kxk_nearest_signed(x, y, bit)
            _, shell_coords = turtleShell.get_shell_coords(x, y, shells, cell_to_shells)
            found = turtleShell.find_corresponding_val(shell_coords, decimals[i], nacp_coord[i], mode)
            nacp_coord[i] = found
            
    return nacp_coord

In [196]:
def data_extract_process(nacp_coord, mode="8N"):
    extracted_data = ""
    data_bits = ""

    for i, (x, y) in enumerate(nacp_coord):
        val = turtleShell.get_hex_matrix_value(x, y, mode=mode)
        bit = 3 if mode == "8N" else 4
        bits = format(val & ((1 << bit) - 1), f'0{bit}b')
        data_bits += bits

        while len(data_bits) >= 8:
            byte = data_bits[:8]
            char_val = int(byte, 2)
            if char_val == 0: # Null terminator ASCII
                return extracted_data
            try:
                char = chr(char_val)
                extracted_data += char
            except:
                return extracted_data
            data_bits = data_bits[8:]

    return extracted_data

In [197]:
def encode(image_path, message_bits):
    sorted_coeffs = get_quantized_coefficients(image_path)
    nacp_coords = get_nacp(sorted_coeffs)
    print(nacp_coords)
    print(f"NACP Length: {len(nacp_coords)}")
    print(f"Total EC: {len(nacp_coords * 3)}")
    modified_nacp_coords = data_hiding_process(message_bits, nacp_coords, mode="8N")
    modified_coeffs = replace_nacp(sorted_coeffs, modified_nacp_coords)
    construct_stego_file(image_path, modified_coeffs)
    print("Data embedding completed.")

def encode_2(image_path, data, qf):
    image = Image.open(image_path).convert('L')
    stegoimg = image.copy()
    img_arr = np.array(stegoimg)
    q_mat = FD.custom_q_mat(qf)
    sorted_coefficients = FD.transform_to_freq(img_arr, q_mat)
    nacp_coords = get_nacp(sorted_coefficients)
    modified_nacp_coords = data_hiding_process(data, nacp_coords)
    modified_coeffs = replace_nacp(sorted_coefficients, modified_nacp_coords)
    np.save("modified_coefficients.npy", modified_coeffs)

def encode_4(image_path, secret_data, qf):
    sorted_coeffs = get_compress_coeff(image_path, qf)
    nacp_coords = get_nacp(sorted_coeffs)
    print(f"NACP Length: {len(nacp_coords)}")
    print(f"Total EC: {len(nacp_coords * 3)}")
    modified_nacp_coords = data_hiding_process(secret_data, nacp_coords, mode="8N")
    modified_coeffs = replace_nacp(sorted_coeffs, modified_nacp_coords)
    construct_stego_file_2(image_path, modified_coeffs, qf)
    
# Proposed Method - Adaptive Payload in Multi Turtle Shell Embedding
def encode_3(image_path, secret_data):
    _, smoothness_score = block_smoothness(image_path)
    smoothness_list = sorted(smoothness_score, key=lambda x: -x[1])
    secret_data += '\0'
    data_bin = ''.join(format(ord(c), '08b') for c in secret_data)
    lendata = len(data_bin)

    _, shells, cell_to_shells = turtleShell.init(mode="8N")
    _, shells_17, cell_to_shells_17 = turtleShell.init(mode="17N")
    
    im = jpeglib.read_dct(image_path)
    for (block_i, block_j), score in smoothness_list:
        # print(f"Processing block ({block_i}, {block_j}) with smoothness score {score}")
        if lendata <= 0: break
        N = 8 if score <= threshold else 17
        t = 3 if N == 8 else 4
        mode = f"{N}N"
        block = im.Y[block_i, block_j]
        zigzag_coeffs = zigzag(block)
        ac_coeffs = zigzag_coeffs[1:]
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]
        choosen_shells = shells if N == 8 else shells_17
        choosen_cell_to_shells = cell_to_shells if N == 8 else cell_to_shells_17
        for idx in range(0, len(non_zero_indices) - 1, 2):
            if lendata <= 0: break
            x = int(ac_coeffs[non_zero_indices[idx]])
            y = int(ac_coeffs[non_zero_indices[idx + 1]])
            bits = data_bin[:t].ljust(t, '0') 
            data_bin = data_bin[t:]
            lendata -= t
            target_val = int(bits, 2)
            val_int = turtleShell.get_hex_matrix_value(x, y, mode=mode)
            if target_val != val_int:
                _, shell_coords = turtleShell.get_shell_coords(x, y, choosen_shells, choosen_cell_to_shells)
                x, y = turtleShell.find_corresponding_val(shell_coords, target_val, (x, y), mode=mode)
            ac_coeffs[non_zero_indices[idx]] = float(x)
            ac_coeffs[non_zero_indices[idx + 1]] = float(y)

        zigzag_coeffs[1:] = ac_coeffs
        zigzag_coeffs[0] = block[0, 0]  
        im.Y[block_i, block_j] = inverse_zigzag(zigzag_coeffs, 8, 8)

    output_path = "stego-images/stego_" + image_path.split("/")[-1]
    print(f"Data embedding completed. Stego image saved to {output_path}")
    im.write_dct(output_path)

In [198]:
def decode(stego_image_path):
    sorted_coeffs = get_quantized_coefficients(stego_image_path)
    nacp_coords = get_nacp(sorted_coeffs)
    print(nacp_coords)
    extracted_data = data_extract_process(nacp_coords, mode="8N")
    return extracted_data

def decode_2(stego_file):
    modified_coeffs = np.load(stego_file, allow_pickle=True)
    nacp_coords = get_nacp(modified_coeffs)
    extracted_data = data_extract_process(nacp_coords)
    return extracted_data

def decode_4(stego_image_path, qf):
    sorted_coeffs = get_compress_coeff(stego_image_path, qf)
    nacp_coords = get_nacp(sorted_coeffs)
    extracted_data = data_extract_process(nacp_coords, mode="8N")
    return extracted_data

# Proposed Method - Adaptive Payload in Multi Turtle Shell Embedding
def decode_3(stego_image_path):
    _, smoothness_score = block_smoothness(stego_image_path)
    smoothness_list = sorted(smoothness_score, key=lambda x: -x[1])
    im = jpeglib.read_dct(stego_image_path)
    bitstream = ""
    decoded_text = ""
    for (block_i, block_j), score in smoothness_list:
        N = 8 if score <= threshold else 17
        t = 3 if N == 8 else 4
        mode = f"{N}N"
        block = im.Y[block_i, block_j]
        zigzag_coeffs = zigzag(block)
        ac_coeffs = zigzag_coeffs[1:]
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]

        for idx in range(0, len(non_zero_indices) - 1, 2):
            x = int(ac_coeffs[non_zero_indices[idx]])
            y = int(ac_coeffs[non_zero_indices[idx + 1]])
            val = turtleShell.get_hex_matrix_value(x, y, mode=mode)
            bits = format(val, f"0{t}b")
            bitstream += bits
            while len(bitstream) >= 8:
                byte = bitstream[:8]
                bitstream = bitstream[8:]
                char_val = int(byte, 2)
                if char_val == 0:   # Null terminator
                    return decoded_text
                decoded_text += chr(char_val)
    return decoded_text

In [199]:
def read_text_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
    return content

In [200]:
convert_tiff_to_jpeg("cover-images/misc/boat.512.tiff", "cover-images/boat_qf50.jpeg", quality=50)

Converted cover-images/misc/boat.512.tiff to cover-images/boat_qf50.jpeg with quality 50


In [201]:
pay_size = 5
cover_folder = "cover-images/"
stego_folder = "stego-images/"
payload_folder = "payload/"
cover_image_path = f"baboon_qf50.jpeg"
stego_image_path = f"stego_baboon_qf50.jpeg"
data = read_text_file(f"{payload_folder}{pay_size}Kb.txt")
# encode_4(f"{cover_folder}{cover_image_path}", data, 50)
encode(f"{cover_folder}{cover_image_path}", data)

secret_data = decode(f"{stego_folder}{stego_image_path}")
# secret_data = decode_4(f"{stego_folder}{stego_image_path}", 50)
print("Extracted Data:", secret_data) 

[(77, -24), (36, -80), (32, 84), (-26, 28), (36, 34), (-64, 95), (-24, -40), (52, 24), (80, -58), (51, -60), (57, 55), (80, 62), (77, -12), (126, -24), (-30, 16), (-65, 42), (36, -34), (-38, -24), (-80, 130), (-66, -36), (40, 80), (110, 48), (84, 84), (40, -32), (70, -26), (-14, 18), (17, 16), (38, -48), (-40, 48), (-22, -24), (40, 57), (77, 48), (-28, -24), (-150, -176), (168, -65), (-42, -54), (-34, 64), (-19, -130), (120, -66), (-37, 87), (-40, 58), (-51, -55), (-22, -60), (-70, 72), (-20, -64), (14, -65), (42, -17), (-16, 19), (-48, 40), (-120, 22), (44, -24), (-37, 58), (-102, 55), (55, 56), (-80, 87), (-154, -84), (-140, -100), (-84, 52), (42, 18), (-68, 80), (-95, 24), (40, -26), (-88, 44), (74, 51), (57, -56), (-81, -87), (88, -48), (154, -72), (30, 98), (-91, 18), (51, -16), (-19, -26), (48, 44), (-44, -24), (-36, 29), (51, -64), (68, 109), (-44, -24), (28, 12), (-150, -32), (56, -26), (70, -54), (17, 64), (-57, -48), (80, 96), (-66, 24), (37, -58), (58, -51), (60, 64), (-72, 

In [202]:
# Test performance metrics
cover_folder = "cover-images/"
stego_folder = "stego-images/"
payload_folder = "payload/"
cover_image_path = f"baboon_qf50.jpeg"
stego_image_path = f"stego_baboon_qf50.jpeg"
psnr_value = psnr(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
fsi_value = fsi(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
ssim_value = ssim(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
print(f"PSNR: {psnr_value} dB")
print(f"FSI: {fsi_value}")
print(f"SSIM: {ssim_value}")

Size cover: 128160
Size stego: 128092


PSNR: 66.44470784565227 dB
FSI: -68.0
SSIM: 0.9999953296250222


In [203]:
def compare_spatial_frequency(cover_image_path, stego_image_path):
    cover = np.array(Image.open(cover_image_path).convert('L'), dtype=np.float64)
    stego = np.array(Image.open(stego_image_path).convert('L'), dtype=np.float64)

    spatial_difference = np.abs(cover - stego) ** 2

    cover_freq = np.fft.fft2(cover)
    stego_freq = np.fft.fft2(stego)
    freq_difference = np.abs(cover_freq - stego_freq)

    return freq_difference, spatial_difference

In [204]:
freq_diff, spatial_diff = compare_spatial_frequency(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
print("Frequency Difference:")
for row in freq_diff:
    print(row)
print("Spatial Difference:")
for row in spatial_diff:
    print(row)
print("Max spatial diff:", spatial_diff.max())
print("Mean spatial diff:", spatial_diff.mean())

Frequency Difference:
[ 45.          73.23469889  33.76162975  49.37690797  58.26788354
  24.55425873  28.45517363   9.42833426  20.52955473  55.18563485
  15.09082106  43.01957326  25.38535451  22.45731249  20.71639802
  51.37047273  42.47974488  68.22472577  42.99609252   8.73765076
  42.97712123  44.34914834  43.10789429  71.61282902  67.06486183
  60.82712059  17.60818504  20.33650883  23.94191934  10.18697495
   4.71413405  32.12790401  26.74830868  24.65981057  46.09692673
  57.63631894  75.70752848  85.63910391  39.60157576  28.35336812
  69.12412098  20.02988944 104.90243908  19.31641221  81.09260084
  29.50624683  68.87758993  51.97722178  45.88379551  13.3188349
 116.75694106 104.96707014  52.13523607  49.53085198  46.05035794
  55.87827089  50.14237801  53.59965043  12.11772449  75.4510649
 175.73019303  83.7605777  159.05625371  81.45320971  64.17448818
  35.39097279  62.5151309  128.22626376 114.21899215 105.20448338
  49.68773004  56.83242014  58.65057719  38.38314584 143

In [205]:
max_pixel = 255.0
psnr_value = 20 * log10(max_pixel / sqrt(spatial_diff.mean()))
print("PSNR calculated from spatial difference:", psnr_value, "dB")

PSNR calculated from spatial difference: 66.44470784565227 dB


In [206]:
image_base = "cover-images/baboon.jpeg"
image_qf50 = "cover-images/baboon_qf50.jpeg"
image_qf70 = "cover-images/baboon_qf70.jpeg"
image_qf90 = "cover-images/baboon_qf90.jpeg"
change_image_QF(image_base, 50)
change_image_QF(image_base, 60)
change_image_QF(image_base, 70)
change_image_QF(image_base, 80)
change_image_QF(image_base, 90)

coeff_qf50 = get_quantized_coefficients(image_qf50)
coeff_qf70 = get_quantized_coefficients(image_qf70)

print("Coeff QF 50")
for row in coeff_qf50:
    print(row)

print("Coeff QF 70")
for row in coeff_qf70:
    print(row)
# freq_diff, spatial_diff = compare_spatial_frequency(f"{image_qf50}", f"{image_qf70}")
# print("Frequency Difference:")
# for row in freq_diff:
#     print(row)
# print("Spatial Difference:")
# for row in spatial_diff:
#     print(row)
# print("Max spatial diff:", spatial_diff.max())
# print("Mean spatial diff:", spatial_diff.mean())

Coeff QF 50
[-384.   77.  -24.    0.   36.  -80.   32.   84.  -26.   28.   36.   34.
  -64.   95.  -24.  -40.   52.   24.    0.    0.    0.    0.    0.    0.
    0.   80.  -58.   51.    0.  -60.   57.    0.    0.    0.    0.    0.
    0.    0.    0.    0.    0.    0.   55.    0.   80.    0.    0.    0.
    0.    0.    0.    0.    0.   62.    0.    0.    0.    0.    0.    0.
    0.    0.    0.    0.]
[-384.   77.  -12.  126.  -24.  -30.   16.    0.  -65.   42.   36.  -34.
    0.  -38.  -24.  -80.  130.    0.  -66.    0.    0.    0.  -36.    0.
    0.   40.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
    0.    0.    0.    0.    0.    0.    0.    0.   80.    0.    0.    0.
    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
    0.    0.    0.    0.]
[-416.  110.   48.   84.   84.   40.  -32.   70.  -26.  -14.   18.   17.
   16.   38.  -48.  -40.    0.   48.    0.  -22.  -24.    0.    0.    0.
    0.   40.    0.    0.    0.    0.   57.    0.    0.    0.